In [1]:
import pandas as pd
import numpy as np
import scanpy as sc
from collections import Counter
from scipy import stats
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from collections import defaultdict
import math
from pathlib import Path
import os
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import scanpy as sc
from collections import Counter
import csv
from sklearn.model_selection import StratifiedKFold
import json, math, os, pathlib

In [14]:
file1 = "/Users/chanyue/Downloads/Asian_Immune_MemoryB_Data.h5ad"
celltype = "memory B cell"

# file1 = "/Users/chanyue/Downloads/Asian_Immune_T_cells.h5ad"
# celltype = "memory T cell"

# file1 = "/Users/chanyue/Downloads/Asian_Immune_ThymusCD4TMonocytes.h5ad"
# celltype = "thymus-derived CD4-positive"

# file1 = "/Users/chanyue/Downloads/Asian_Immune_naive_B_adata.h5ad"
# celltype = "native B cell"

# file1 = "/Users/chanyue/Downloads/Asian_Immune_B_cells.h5ad"
# celltype = "B cell"

In [15]:
adata = sc.read(file1)

In [16]:
adata.obs["cell_type"]

index
GTGTGCGAGGCTAGCA-SG_B1_L1     memory B cell
TCAGCTCAGAAGATTC-SG_B1_L1     memory B cell
CTGCGGACACGAAGCA-SG_B1_L1     memory B cell
GGCTCGACATAACCTG-SG_B1_L1     memory B cell
GACTAACAGTACGCCC-SG_B1_L1     memory B cell
                                  ...      
GGTGTTAAGACAGAGA-KR_B12_L2    memory B cell
GTCGGGTCAAACGCGA-KR_B12_L2    memory B cell
CATATTCTCATTCACT-KR_B12_L2    memory B cell
CCAATCCCACAAGACG-KR_B12_L2    memory B cell
GACTGCGCACTTAAGC-KR_B12_L2    memory B cell
Name: cell_type, Length: 28247, dtype: category
Categories (1, object): ['memory B cell']

In [17]:
sc.pp.filter_genes(adata, min_cells=3)
sc.pp.filter_cells(adata, min_genes=200)

In [18]:
sc.pp.normalize_total(adata, target_sum=1e4, inplace=True)

In [19]:
sc.pp.log1p(adata)

In [20]:
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
adata = adata[:, adata.var.highly_variable]

In [21]:
adata

View of AnnData object with n_obs × n_vars = 28247 × 1028
    obs: 'reference_genome', 'gene_annotation_version', 'alignment_software', 'intronic_reads_counted', 'library_id', 'assay_ontology_term_id', 'sequenced_fragment', 'cell_number_loaded', 'institute', 'is_primary_data', 'cell_type_ontology_term_id', 'author_cell_type', 'sample_id', 'sample_preservation_method', 'tissue_ontology_term_id', 'development_stage_ontology_term_id', 'sample_collection_method', 'donor_BMI_at_collection', 'tissue_type', 'suspension_derivation_process', 'suspension_enriched_cell_types', 'cell_viability_percentage', 'suspension_uuid', 'suspension_type', 'donor_id', 'self_reported_ethnicity_ontology_term_id', 'donor_living_at_sample_collection', 'disease_ontology_term_id', 'sex_ontology_term_id', 'Country', 'nCount_RNA', 'nFeature_RNA', 'TCR_VDJdb', 'TCRa_V_gene', 'TCRa_D_gene', 'TCRa_J_gene', 'TCRa_C_gene', 'TCRb_V_gene', 'TCRb_D_gene', 'TCRb_J_gene', 'TCRb_C_gene', 'TCR_Clonality', 'TCR_Clone_ID', 'BCR_VDJ

In [22]:
import inflect

p = inflect.engine()

In [23]:
# ─── Pull data from AnnData ─────────────────────────────────────────────────────
genes   = np.asarray(adata.var_names)
X       = adata.X
# ages    = adata.obs["age"].values
ages    = [int(x.split("-")[0]) for x in adata.obs["development_stage"].values]
# genders = adata.obs["sex"].values
# classes = adata.obs["cell_ontology_class"].values
classes = adata.obs["cell_type"]
# tissues = adata.obs["tissue"].values
tissues = np.array(["blood"] * adata.n_obs)

print(Counter(ages))

Counter({33: 1444, 39: 1170, 34: 1167, 21: 1159, 41: 1151, 26: 1075, 43: 1009, 30: 935, 48: 924, 31: 885, 37: 868, 40: 852, 23: 847, 36: 838, 49: 815, 28: 807, 47: 748, 44: 741, 38: 737, 56: 690, 45: 688, 46: 680, 29: 633, 58: 619, 53: 599, 35: 556, 42: 531, 61: 500, 55: 480, 59: 457, 50: 421, 52: 405, 27: 402, 62: 389, 66: 388, 32: 374, 60: 331, 57: 318, 54: 309, 51: 305})


In [24]:
# ─── Split config ───────────────────────────────────────────────────────────────
N_SPLITS   = 11
SPLIT_SIZE = math.ceil(adata.n_obs / N_SPLITS)
OUT_DIR    = "fine_tune_chunks"
os.makedirs(OUT_DIR, exist_ok=True)

INSTRUCTION = "We will give you a sequence of scRNA transcriptomics data of a single cell. We would like to predict the age of the cell based on the gene name. The age is usually decided by combination of genes. The following are the gene names (Genes), gender (Gender), cell type (Class), and tissue, separated by the keywords Genes, Gender, Class, and Tissue. Different cell type (Class) has different combinations from each other."

def bag_of_words(counts_row: np.ndarray, gene_names: np.ndarray) -> str:
    """Convert a vector of counts to 'gene english(count)' tokens separated by space."""
    counts_row = np.maximum(np.round(counts_row).astype(int), 0)
    tokens = [f"{gene} {p.number_to_words(count)}"
        for gene, count in zip(gene_names, counts_row)
        if count > 0]
    return " ".join(tokens)

# ─── Build chunk files in **Alpaca** format ─────────────────────────────────────
# check length 
lengths = []
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
for split_idx, (_, split_indices) in enumerate(skf.split(np.zeros(len(ages)), ages)):
    records = []
    for i in split_indices:
        row = X[i].toarray().ravel() if not isinstance(X, np.ndarray) else X[i]
        lengths.append(len(bag_of_words(row, genes)))
        cell_input = (
            f"Genes: {bag_of_words(row, genes)}\n"
#             f"Gender: {genders[i]}\n"
            f"Class: {classes[i]}\n"
            f"Tissue: {tissues[i]}"
        )
    
        records.append({
            "instruction": INSTRUCTION,
            "input": cell_input,
            "output": str(ages[i])
        })
    fname = os.path.join(OUT_DIR, f"Asian_Immune_{celltype}_cell_data_part_{split_idx+1}.json")
    with open(fname, "w", encoding="utf-8") as f:
        json.dump(records, f, indent=2, ensure_ascii=False)

    print(f"Wrote {len(records):>5} samples → {fname}")

Wrote  2568 samples → fine_tune_chunks/Asian_Immune_memory B cell_cell_data_part_1.json
Wrote  2568 samples → fine_tune_chunks/Asian_Immune_memory B cell_cell_data_part_2.json
Wrote  2568 samples → fine_tune_chunks/Asian_Immune_memory B cell_cell_data_part_3.json
Wrote  2568 samples → fine_tune_chunks/Asian_Immune_memory B cell_cell_data_part_4.json
Wrote  2568 samples → fine_tune_chunks/Asian_Immune_memory B cell_cell_data_part_5.json
Wrote  2568 samples → fine_tune_chunks/Asian_Immune_memory B cell_cell_data_part_6.json
Wrote  2568 samples → fine_tune_chunks/Asian_Immune_memory B cell_cell_data_part_7.json
Wrote  2568 samples → fine_tune_chunks/Asian_Immune_memory B cell_cell_data_part_8.json
Wrote  2568 samples → fine_tune_chunks/Asian_Immune_memory B cell_cell_data_part_9.json
Wrote  2568 samples → fine_tune_chunks/Asian_Immune_memory B cell_cell_data_part_10.json
Wrote  2567 samples → fine_tune_chunks/Asian_Immune_memory B cell_cell_data_part_11.json
